# 7- Random Forest Classifier

In [38]:
# Librerias para métricas de rendimiento de los modelos

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay

In [39]:
from sklearn.ensemble import RandomForestClassifier

In [40]:
# Instancia del modelo
rf = RandomForestClassifier()

# Hiper-parametros
param_grid_rf = {
    'n_estimators': [32, 64, 128, 256],
    'max_features': [2,3,4],
    'bootstrap': [True, False]
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5)

In [41]:
grid_rf.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'bootstrap': [True, False], 'max_features': [2, 3, 4],
                         'n_estimators': [32, 64, 128, 256]})

In [42]:
y_pred_rf = grid_rf.predict(X_test_scaled)
y_pred_rf_train = grid_rf.predict(X_train_scaled)

# Mejores parametros del modelo
grid_rf.best_params_

{'bootstrap': False, 'max_features': 4, 'n_estimators': 32}

De momento, validaremos su Accuracy y demás, con el set de entrenamiento y también el de test para ver si el modelo tiene overfitting

In [43]:
print('='*100)
print('TRAIN')
print('='*100)
print(classification_report(y_train, y_pred_rf_train))

print('='*100)
print('TEST')
print('='*100)
print(classification_report(y_test, y_pred_rf))

TRAIN
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        94
           1       1.00      1.00      1.00       706

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800

TEST
              precision    recall  f1-score   support

           0       0.89      0.70      0.78        23
           1       0.96      0.99      0.97       177

    accuracy                           0.95       200
   macro avg       0.93      0.84      0.88       200
weighted avg       0.95      0.95      0.95       200



Como hemos dectetado overfitting (aparentemente es que haya profundizado demasiado en los arboles y aprendió los patrones exactos de los datos de entrenamiento, ya que los predice todos tal cual, sin margen de error) en el modelo de Random Forest, vamos a proceder a ajustar sus hiper-parametros y, aplicar el 'balance' de las clases, que como ya detectamos anteriormente (en el EDA) que hay un desbalance claro con los clientes que ABANDONAR y con los que no ABANDONARON, este ajuste hará que el modelo tenga más peso en la clase minorista.

In [44]:
balance_clases = df.groupby('Churn')['Churn'].count()
balance_clases

,Churn
Churn,
No,117
Yes,883


In [45]:
from sklearn.metrics import make_scorer, f1_score

m_rf = RandomForestClassifier(class_weight='balanced', random_state=42)
f1_clase_0 = make_scorer(f1_score, pos_label=0)

entrenamiento = []

def rf_ajustado(modelo, X_entrenamiento, X_prueba, y_entrenamiento):

  parametros = {
      'n_estimators': [100,200],
      'max_depth': [3,4,5],
      'min_samples_split': [5, 10, 20],
      'min_samples_leaf': [2, 5, 20],
      'max_features': ['sqrt','log2']
  }

  # GridSearchCV con 5 k-folds
  grid_rf_ajustado = GridSearchCV(modelo, parametros, cv=5, scoring=f1_clase_0)

  # Entrenamiento del modelo
  grid_rf_ajustado.fit(X_entrenamiento, y_entrenamiento)

  print(f'Mejores hiperparametros: {grid_rf_ajustado.best_params_}')

  # Prediccion de TEST
  predict_test = grid_rf_ajustado.predict(X_prueba)

  return grid_rf_ajustado

In [64]:
grid_rf_ajustado = rf_ajustado(modelo=m_rf, X_entrenamiento=X_train_scaled, X_prueba=X_test_scaled,
            y_entrenamiento=y_train)

Mejores hiperparametros: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}


Luego de balancear el modelo y ajustar sus hiper-parametros a unos más limitados, ya el modelo no presenta overfitting y las métricas de rendimiento están mucho mejor que el modelo anterior el cual no está balanceado... si bien las métricas mejoraron mucho y ya no hay overfitting, tenemos una pequeña 'brecha' en la **Precision** en la cual el modelo podrá clasificar una clase (de la clase minorista, los que no abandonan), en cambio por un **Recall** muy bueno.

In [47]:
print("--- MÉTRICAS EN PRUEBA ---")
print(classification_report(y_test, grid_rf_ajustado.predict(X_test_scaled)))

--- MÉTRICAS EN PRUEBA ---
              precision    recall  f1-score   support

           0       0.71      0.96      0.81        23
           1       0.99      0.95      0.97       177

    accuracy                           0.95       200
   macro avg       0.85      0.95      0.89       200
weighted avg       0.96      0.95      0.95       200



# 8- Modelo de Regresión Logistica

In [48]:
log_model = LogisticRegression(class_weight='balanced', random_state=42)

#Entrenamiento
log_model.fit(X_train_scaled, y_train)

# Prediccion con datos de prueba
y_pred_lr = log_model.predict(X_test_scaled)

# Predicción con datos de entrenamiento
y_pred_lr_train = log_model.predict(X_train_scaled)

Las métricas de rendimiento arrojan buenos resultados para el modelo. Un Precision de 0.43 y un Recall de 1.00 (en NO) está muy bien, esto aplicandole claro está la clase de 'balanced' al modelo.

In [49]:
print('===== TRAIN ======')
print(classification_report(y_train, y_pred_lr_train))

print('===== TEST ======')
print(classification_report(y_test, y_pred_lr))

===== TRAIN ======
              precision    recall  f1-score   support

           0       0.45      0.99      0.62        94
           1       1.00      0.84      0.91       706

    accuracy                           0.86       800
   macro avg       0.72      0.91      0.77       800
weighted avg       0.93      0.86      0.88       800

===== TEST ======
              precision    recall  f1-score   support

           0       0.43      1.00      0.60        23
           1       1.00      0.82      0.90       177

    accuracy                           0.84       200
   macro avg       0.71      0.91      0.75       200
weighted avg       0.93      0.84      0.87       200



# 9- Modelo con KNE

Con KNE, no podemos aplicar 'class_weight=balanced', con este modelo vamos a aplicar 'SMOTE', algoritmo para equilibrar un o desbalanceo en las **clases**: una clase con un peso muchísimo mayor a otro. Este algoritmo nos ayudará a equilibrar ese factor.

In [50]:
# Instancia de smote
smote = SMOTE(random_state=42)

# Versión balanceada de los datos
X_train_balanceado, y_train_balanceado = smote.fit_resample(X_train_scaled, y_train)

# Verificación de las clases balanceadas
print(f'Distribución original:{y_train.value_counts(normalize=True).to_dict()}')
print(f'Distribución balanceada: {y_train_balanceado.value_counts(normalize=True).to_dict()}')

Distribución original:{1: 0.8825, 0: 0.1175}
Distribución balanceada: {0: 0.5, 1: 0.5}


In [51]:
# Instancia del modelo
kne = KNeighborsClassifier()

# Ajuste de hiperparametros
param_grid_kne = {
    'n_neighbors': [3,5,7,9],
    'weights': ['uniform', 'distance']
}

grid_kne = GridSearchCV(kne, param_grid_kne, cv=5, scoring=f1_clase_0)

# Entrenamiento con los datos balanceados con SMOTE
grid_kne.fit(X_train_balanceado, y_train_balanceado)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 7, 9],
                         'weights': ['uniform', 'distance']},
             scoring=make_scorer(f1_score, response_method='predict', pos_label=0))

In [52]:
y_pred_kne = grid_kne.predict(X_test_scaled)

Hemos obtuvido las mejores métricas con el modelo de KNE, con un Accuracy(0.91) y un Recall y Precision mejor a los modelos de RLogistica y RF.

In [53]:
print(classification_report(y_test, y_pred_kne))

              precision    recall  f1-score   support

           0       0.58      0.83      0.68        23
           1       0.98      0.92      0.95       177

    accuracy                           0.91       200
   macro avg       0.78      0.87      0.81       200
weighted avg       0.93      0.91      0.92       200

